# Questão 6 - Previsão de Demanda: Bússola de Bordo 702

**Premissas obrigatórias:**
- Treino: até 31/12/2025
- Teste: 1º trimestre de 2026
- Granularidade mensal
- Produto: "Bússola de Bordo 702"
- Status: sem filtro; todos os pedidos são mantidos
- Avaliação: walk-forward mensal (um passo à frente)

In [12]:
import pandas as pd
from src.db import get_engine

engine = get_engine()


## 1. Dataset unificado: identificação do produto

In [13]:
products = pd.read_sql("SELECT * FROM products WHERE name = 'Bússola de Bordo 702'", engine)
print("Cadastros encontrados:")
display(products[['id', 'name', 'description', 'brand_id', 'category_id', 'is_active', 'created_at']])

bussola_product_ids = products['id'].tolist()

Cadastros encontrados:


,id,name,description,brand_id,category_id,is_active,created_at
0,74,Bússola de Bordo 702,Bússola magnética líquida com iluminação,12,8,True,2025-01-27 17:41:23
1,240,Bússola de Bordo 702,Bússola magnética líquida com iluminação,8,7,True,2026-06-22 00:02:10


In [14]:
variants = pd.read_sql(
    f"SELECT * FROM product_variants WHERE product_id IN ({','.join(map(str, bussola_product_ids))})",
    engine
)

for pid in bussola_product_ids:
    variantes_pid = variants[variants['product_id'] == pid]['id'].tolist()
    vendas_pid = pd.read_sql(
        f"SELECT MIN(o.created_at) AS primeira_venda, MAX(o.created_at) AS ultima_venda, COUNT(*) AS qtd_vendas "
        f"FROM order_items oi JOIN orders o ON o.id = oi.order_id "
        f"WHERE oi.product_variant_id IN ({','.join(map(str, variantes_pid))})",
        engine
    )
    data_criacao = products[products['id'] == pid]['created_at'].values[0]
    print(f"Produto {pid}, criado em: {data_criacao}")
    display(vendas_pid)
    print()

Produto 74, criado em: 2025-01-27T17:41:23.000000000


,primeira_venda,ultima_venda,qtd_vendas
0,2020-01-03 22:58:57,2026-12-31 21:59:40,330



Produto 240, criado em: 2026-06-22T00:02:10.000000000


,primeira_venda,ultima_venda,qtd_vendas
0,2020-01-11 11:20:28,2026-12-27 11:13:12,142


**Produto duplicado:** existem dois cadastros com nome e descrição idênticos (IDs 74 e 240),
ambos ativos, mas com `brand_id` e `category_id` diferentes. Como mostrado acima, os dois IDs
têm vendas registradas *antes* de suas próprias
datas de criação, portanto `created_at` não permite desambiguá-los de forma confiável. Como o
enunciado identifica o produto pelo nome, a regra adotada é agregar as vendas das variantes dos
dois cadastros. Essa é uma hipótese analítica explícita; ela não afirma que os cadastros sejam
necessariamente o mesmo item físico.

In [15]:
bussola_variant_ids = variants['id'].tolist()

query_sem_filtro = f"""
SELECT oi.quantity, o.created_at, o.status
FROM order_items oi
JOIN orders o ON o.id = oi.order_id
WHERE oi.product_variant_id IN ({','.join(map(str, bussola_variant_ids))})
"""
vendas_sem_filtro = pd.read_sql(query_sem_filtro, engine)

print("Distribuição de status nos itens da Bússola de Bordo 702:")
display(vendas_sem_filtro['status'].value_counts())

print("\nTotal de unidades (sem filtro):", vendas_sem_filtro['quantity'].sum())
print("Total de unidades (só paid/confirmed):",
      vendas_sem_filtro[vendas_sem_filtro['status'].isin(['paid', 'confirmed'])]['quantity'].sum())
print("Unidades em cancelled/draft:",
      vendas_sem_filtro[vendas_sem_filtro['status'].isin(['cancelled', 'draft'])]['quantity'].sum())

Distribuição de status nos itens da Bússola de Bordo 702:


status
paid         341
confirmed     67
cancelled     41
draft         23
Name: count, dtype: int64


Total de unidades (sem filtro): 2543
Total de unidades (só paid/confirmed): 2200
Unidades em cancelled/draft: 343


In [16]:
status_incluidos = sorted(vendas_sem_filtro['status'].dropna().unique())

print("Regra adotada: todos os status foram mantidos.")
print("Status incluídos:", ', '.join(status_incluidos))

Regra adotada: todos os status foram mantidos.
Status incluídos: cancelled, confirmed, draft, paid


**Regra de status (definida antes da avaliação):** o enunciado não solicita a exclusão de
nenhum status. Por isso, todos os `order_items` ligados ao produto são considerados na demanda,
inclusive os associados a pedidos `cancelled` e `draft`. A distribuição acima é apenas um
diagnóstico da base; o erro do período de teste não é usado para escolher esse tratamento.

In [17]:
vendas = vendas_sem_filtro.copy()
vendas['created_at'] = pd.to_datetime(vendas['created_at'])
vendas['mes'] = vendas['created_at'].dt.to_period('M')

vendas_mensais = vendas.groupby('mes')['quantity'].sum().sort_index()
idx_completo = pd.period_range(vendas_mensais.index.min(), vendas_mensais.index.max(), freq='M')
vendas_mensais = vendas_mensais.reindex(idx_completo, fill_value=0)

print("Vendas mensais (últimos 12 meses):")
vendas_mensais.tail(12)

Vendas mensais (últimos 12 meses):


2026-01    79
2026-02    68
2026-03    60
2026-04    36
2026-05    36
2026-06    38
2026-07    24
2026-08    25
2026-09    56
2026-10    29
2026-11    62
2026-12    64
Freq: M, Name: quantity, dtype: int64

In [18]:
print("Nulos em quantity:", vendas['quantity'].isna().sum())
print("Nulos em created_at:", vendas['created_at'].isna().sum())
print("Quantity <= 0:", (vendas['quantity'] <= 0).sum())

orders_check = pd.read_sql("SELECT id FROM orders", engine)
oi_check = pd.read_sql(
    f"SELECT id, order_id, product_variant_id FROM order_items "
    f"WHERE product_variant_id IN ({','.join(map(str, bussola_variant_ids))})",
    engine
)
orphan_orders = oi_check[~oi_check['order_id'].isin(orders_check['id'])]
orphan_variants = oi_check[~oi_check['product_variant_id'].isin(variants['id'])]
print("order_items com order_id órfão:", len(orphan_orders))
print("order_items com product_variant_id órfão:", len(orphan_variants))
print("order_items.id duplicado:", oi_check['id'].duplicated().sum())

q1, q3 = vendas['quantity'].quantile([0.25, 0.75])
iqr = q3 - q1
outliers = vendas[vendas['quantity'] > q3 + 3 * iqr]
print(f"\nOutliers em quantity (> Q3 + 3xIQR = {q3 + 3*iqr:.1f}):", len(outliers))
print("\nEstatística descritiva de quantity:")
vendas['quantity'].describe()

Nulos em quantity: 0
Nulos em created_at: 0
Quantity <= 0: 0
order_items com order_id órfão: 0
order_items com product_variant_id órfão: 0
order_items.id duplicado: 0

Outliers em quantity (> Q3 + 3xIQR = 23.0): 0

Estatística descritiva de quantity:


count    472.000000
mean       5.387712
std        2.936717
min        1.000000
25%        3.000000
50%        5.000000
75%        8.000000
max       10.000000
Name: quantity, dtype: float64

Com a regra sem filtro de status, foram verificados: zero nulos, zero registros órfãos
(integridade referencial `order_items` → `orders`/`product_variants`), zero duplicatas e zero
outliers em `quantity` (variando de 1 a 10 unidades por item). Nenhum registro é removido ou
corrigido; a inconsistência entre datas de venda e cadastro permanece documentada acima.

## 2. Baseline: média móvel dos últimos 3 meses

Previsão para o mês M = média das vendas reais dos 3 meses imediatamente anteriores a M.
A avaliação principal é **walk-forward mensal (um passo à frente)**: janeiro/2026 é previsto
com out/nov/dez de 2025; depois de janeiro ser observado, fevereiro é previsto com
nov/dez/jan; após fevereiro ser observado, março usa dez/jan/fev. Assim, a origem avança a
cada mês e nenhuma previsão usa dados do próprio mês ou de meses futuros.

Isso difere de um forecast fixo emitido integralmente em 31/12/2025, no qual os três meses
teriam de ser previstos sem incorporar observações do trimestre. Esse cenário é mostrado ao
final apenas para explicitar a diferença de protocolo, não para selecionar o modelo.

In [19]:
forecast = vendas_mensais.rolling(window=3).mean().shift(1)

forecast.tail(6)

2026-07    36.666667
2026-08    32.666667
2026-09    29.000000
2026-10    35.000000
2026-11    36.666667
2026-12    49.000000
Freq: M, Name: quantity, dtype: float64

## 3. Previsão mensal (Q1 2026)


In [20]:
periodo_teste = pd.period_range('2026-01', '2026-03', freq='M')

resultado = pd.DataFrame({
    'real': vendas_mensais.reindex(periodo_teste),
    'previsto': forecast.reindex(periodo_teste)
})
resultado['previsto_arredondado'] = resultado['previsto'].round().astype(int)

resultado

,real,previsto,previsto_arredondado
2026-01,79,38.666667,39
2026-02,68,53.666667,54
2026-03,60,56.333333,56


## 4. Avaliação: MAE (Mean Absolute Error)


In [21]:
resultado['erro_absoluto'] = (resultado['real'] - resultado['previsto']).abs()

mae = resultado['erro_absoluto'].mean()
soma_previsao = resultado['previsto_arredondado'].sum()

print(f"MAE: {mae:.2f}")
print(f"Soma real Q1 2026: {resultado['real'].sum()}")
print(f"Soma da previsão (arredondada) Q1 2026: {soma_previsao}")

resultado

MAE: 19.44
Soma real Q1 2026: 207
Soma da previsão (arredondada) Q1 2026: 149


,real,previsto,previsto_arredondado,erro_absoluto
2026-01,79,38.666667,39,40.333333
2026-02,68,53.666667,54,14.333333
2026-03,60,56.333333,56,3.666667


## Questão 6.2 - Validação

**Soma total da previsão (arredondada) para o 1º trimestre de 2026: 149 unidades**

(Jan: 39 + Fev: 54 + Mar: 56 = 149; valor real do trimestre: 207 unidades)

## 5. Resposta objetiva

**a. O baseline é adequado para esse produto?**

Parcialmente. O MAE de 19,44 equivale a aproximadamente 28% da demanda mensal média real do
trimestre (69 unidades). O erro diminui de 40,33 unidades em janeiro para 3,67 em março, mas
o resultado ainda é mais adequado como referência inicial do que como previsão final para
decisões de compra.

**b. Uma limitação desse método:**

A média móvel simples não captura sazonalidade nem mudanças bruscas de nível: ela reage com
atraso porque resume apenas os três meses anteriores. O salto observado em janeiro ilustra
essa limitação; no walk-forward, o valor alto passa a influenciar fevereiro e março, reduzindo
o erro sem representar uma capacidade explícita de modelar a mudança.

In [22]:
# Cenário informativo: três previsões emitidas de uma vez em 31/12/2025.
media_fixa_treino = vendas_mensais.loc['2025-10':'2025-12'].mean()

resultado_fixo = pd.DataFrame({
    'real': vendas_mensais.reindex(periodo_teste),
    'previsto_fixo': media_fixa_treino
})
resultado_fixo['erro_absoluto'] = (resultado_fixo['real'] - resultado_fixo['previsto_fixo']).abs()
mae_fixo = resultado_fixo['erro_absoluto'].mean()

print(f"Cenário informativo: forecast fixo em 31/12/2025 (média out/nov/dez = {media_fixa_treino:.2f}):")
display(resultado_fixo)
print(f"MAE do forecast fixo: {mae_fixo:.2f}")
print(f"MAE da avaliação walk-forward principal: {mae:.2f}")

Cenário informativo: forecast fixo em 31/12/2025 (média out/nov/dez = 38.67):


,real,previsto_fixo,erro_absoluto
2026-01,79,38.666667,40.333333
2026-02,68,38.666667,29.333333
2026-03,60,38.666667,21.333333


MAE do forecast fixo: 30.33
MAE da avaliação walk-forward principal: 19.44


## Questão 6.3 - Explique

**1. Como o baseline foi construído?**

Agregando a demanda (soma de `quantity`) por mês, a partir do join entre `order_items` e
`orders` para as variantes dos dois cadastros com o nome solicitado, sem filtro de status.
Na avaliação walk-forward, a previsão de cada mês é a média aritmética simples das vendas
reais dos 3 meses imediatamente anteriores (`rolling(3).mean().shift(1)`).

**2. Como evitou data leakage?**

O `.shift(1)` garante que a previsão do mês M nunca inclua a demanda do próprio mês M. Na
primeira origem, janeiro usa apenas out/nov/dez de 2025, respeitando o corte de treino em
31/12/2025. Depois que janeiro é observado, a origem avança e fevereiro usa nov/dez/jan; após
fevereiro ser observado, março usa dez/jan/fev. Portanto, cada previsão é feita um passo à
frente, sem dados do próprio mês ou de meses futuros.

Um forecast trimestral fixo emitido em 31/12/2025 responderia a outra pergunta operacional:
sem atualizar a janela, repetiria 38,67 unidades para janeiro, fevereiro e março. Seu MAE é
30,33, enquanto o walk-forward tem MAE 19,44. Essa comparação é apenas informativa e não é
usada para escolher o protocolo principal nem a regra de status com base no conjunto de teste.

**3. Uma limitação do modelo proposto:**

Além de não capturar sazonalidade nem mudanças bruscas de nível, o modelo é sensível à
decisão de agregar os dois cadastros com nome idêntico. Se eles representarem itens comerciais
distintos, a série histórica combinará demandas diferentes e o baseline perderá interpretação.